# M-Pesa Float Demand - Exploratory Data Analysis

This notebook performs comprehensive EDA on M-Pesa transaction and float data to understand patterns, trends, and anomalies.

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("✓ Libraries imported successfully")

## 2. Setup Configuration and Database Connection

In [ ]:
import sys
sys.path.insert(0, '..')

from config import settings
from logger import logger
from db import init_db, get_db
from ingestion.cbk_stats import CBKMPesaClient

# Initialize database
try:
    init_db()
    logger.info("Database connected successfully")
except Exception as e:
    logger.warning(f"Database connection failed: {e}")

# Display configuration
print(f"📊 Environment: {settings.ENVIRONMENT}")
print(f"📁 Data Directory: {settings.DATA_DIR}")
print(f"🔍 Forecast Horizon: {settings.FORECAST_HORIZON} days")
print(f"✅ Confidence Level: {settings.CONFIDENCE_LEVEL:.1%}")

## 3. Data Ingestion and Exploration

In [ ]:
# Ingest M-Pesa statistics
cbk_client = CBKMPesaClient()

# Get historical M-Pesa data
mpesa_data = cbk_client.get_mpesa_statistics()
agent_data = cbk_client.get_agent_float_data()
regional_data = cbk_client.get_regional_summary()

print(f"📦 M-Pesa Statistics: {len(mpesa_data)} records")
print(f"👥 Agent Data: {len(agent_data)} records")
print(f"🗺️  Regional Data: {len(regional_data)} regions")
print("\nSample M-Pesa Data:")
print(mpesa_data.head())

In [ ]:
# Summary statistics
print("\n📊 Summary Statistics - M-Pesa Data")
print(mpesa_data.describe())

print("\n📊 Summary Statistics - Agent Data")
print(agent_data.describe())

In [ ]:
# Data quality checks
print("🔍 Data Quality Checks\n")
print("M-Pesa Data:")
print(f"  - Missing values: {mpesa_data.isnull().sum().sum()}")
print(f"  - Date range: {mpesa_data['date'].min()} to {mpesa_data['date'].max()}")
print(f"  - Duplicates: {mpesa_data.duplicated().sum()}")

print("\nAgent Data:")
print(f"  - Missing values: {agent_data.isnull().sum().sum()}")
print(f"  - Unique agents: {agent_data['agent_id'].nunique()}")
print(f"  - Duplicates: {agent_data.duplicated().sum()}")

In [ ]:
# Time series visualization
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Daily Transactions', 'Daily Transaction Value'),
    specs=[[{'secondary_y': False}], [{'secondary_y': False}]]
)

fig.add_trace(
    go.Scatter(x=mpesa_data['date'], y=mpesa_data['total_transactions'],
               name='Transactions', mode='lines',
               line=dict(color='#1f77b4', width=2)),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=mpesa_data['date'], y=mpesa_data['total_value'],
               name='Transaction Value', mode='lines',
               line=dict(color='#ff7f0e', width=2)),
    row=2, col=1
)

fig.update_yaxes(title_text="Transactions", row=1, col=1)
fig.update_yaxes(title_text="Value (KES)", row=2, col=1)
fig.update_xaxes(title_text="Date", row=2, col=1)

fig.update_layout(height=700, title_text="M-Pesa Time Series Trends",
                  hovermode='x unified', showlegend=True)
fig.show()

In [ ]:
# Regional distribution
fig = go.Figure(data=[
    go.Bar(x=regional_data['region'], y=regional_data['total_agents'],
           name='Total Agents', marker_color='#1f77b4'),
    go.Bar(x=regional_data['region'], y=regional_data['avg_daily_transactions'],
           name='Daily Transactions', marker_color='#ff7f0e')
])

fig.update_layout(
    title='Agent Distribution by Region',
    xaxis_title='Region',
    yaxis_title='Count',
    barmode='group',
    height=500,
    hovermode='x unified'
)
fig.show()

## 4. Feature Engineering Pipeline

In [ ]:
from features.feature_engineering import FeatureEngineering
from features.salary_cycle import SalaryCycleAnalyzer
from features.event_calendar import EventCalendar

# Initialize feature engineering
fe = FeatureEngineering()
salary_analyzer = SalaryCycleAnalyzer()
event_calendar = EventCalendar()

# Create time series
series = pd.Series(
    mpesa_data['total_transactions'].values,
    index=pd.to_datetime(mpesa_data['date'])
)

# Generate all features
features_df = fe.create_all_features(series, series.index)

print(f"✓ Created {len(features_df.columns)} features")
print(f"✓ {len(features_df)} records after feature engineering")
print("\nFeature columns:")
print(features_df.columns.tolist())

In [ ]:
# Add financial calendar features
financial_calendar = salary_analyzer.get_financial_calendar(series.index)
event_features = event_calendar.get_event_calendar(series.index)

print("\n💰 Financial Calendar Features:")
print(financial_calendar.head())
print(f"\nSalary days marked: {financial_calendar['is_salary_day'].sum()}")

print("\n📅 Event Calendar Features:")
print(event_features.head())
print(f"Holidays: {event_features['is_holiday'].sum()}")

In [ ]:
# Feature correlation analysis
correlation_matrix = features_df.corr()

# Get top correlations with target
top_features = correlation_matrix['target'].abs().sort_values(ascending=False)[1:11]

fig = go.Figure(data=[
    go.Bar(x=top_features.values, y=top_features.index, orientation='h',
           marker_color='#1f77b4')
])

fig.update_layout(
    title='Top 10 Feature Correlations with Target',
    xaxis_title='Correlation',
    yaxis_title='Feature',
    height=500
)
fig.show()

print("\n📊 Top Feature Correlations:")
print(top_features)

## 5. Seasonality and Trend Analysis

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

# Perform seasonal decomposition
decomposition = seasonal_decompose(series, model='additive', period=30)

fig = make_subplots(
    rows=4, cols=1,
    subplot_titles=('Original', 'Trend', 'Seasonal', 'Residual'),
    vertical_spacing=0.08
)

fig.add_trace(go.Scatter(x=series.index, y=series.values, name='Original',
                         line=dict(color='#1f77b4')), row=1, col=1)
fig.add_trace(go.Scatter(x=decomposition.trend.index, y=decomposition.trend.values, 
                         name='Trend', line=dict(color='#ff7f0e')), row=2, col=1)
fig.add_trace(go.Scatter(x=decomposition.seasonal.index, y=decomposition.seasonal.values, 
                         name='Seasonal', line=dict(color='#2ca02c')), row=3, col=1)
fig.add_trace(go.Scatter(x=decomposition.resid.index, y=decomposition.resid.values, 
                         name='Residual', line=dict(color='#d62728')), row=4, col=1)

fig.update_yaxes(title_text='Transactions', row=1, col=1)
fig.update_layout(height=900, title_text='Seasonal Decomposition (Period=30)',
                  hovermode='x unified', showlegend=True)
fig.show()

## 6. Summary Statistics

print("\n📈 Time Series Statistics\n")
print(f"Mean: {series.mean():.0f}")
print(f"Median: {series.median():.0f}")
print(f"Std Dev: {series.std():.0f}")
print(f"Min: {series.min():.0f}")
print(f"Max: {series.max():.0f}")
print(f"CV: {(series.std() / series.mean()):.2%}")
print(f"\nTrend: {'Upward' if decomposition.trend.iloc[-1] > decomposition.trend.iloc[0] else 'Downward'}")
print(f"Seasonal Strength: {(1 - decomposition.resid.var() / decomposition.seasonal.var()):.2%}")

In [ ]:
print("\n✅ Exploratory Data Analysis Complete!")
print("\nKey Findings:")
print(f"  • Data spans {(series.index[-1] - series.index[0]).days} days")
print(f"  • Strong trend detected in series")
print(f"  • Seasonal patterns with period ~30 days")
print(f"  • {agent_data['region'].nunique()} distinct regions in data")
print("\nReady for model training!")